# Sonic v8.0.1 — dual probe + Phoenix v8.1 judge, v8.0 sign-gate calibration

Dual L40+L46 white-box probe (fused z-scores, no token cap) + Phoenix Wright
v8.1 direct-margin judge (Kimi K3 Liars-enriched, r16 adapter, HP-KR + action routes).
Blended under the unchanged v8.0 sign gate and its frozen judge-scale constants.

## v8.0.1 change from v8.0

Only the judge adapter changes: Phoenix v8.0's competition-only Kimi K3
student is replaced by the Phoenix v8.1 competition-plus-Liars student.
The probe, routes, renderer, direct-margin readout, gate, and binary threshold
are unchanged. `JUDGE_LOGIT_SD=5.638` remains the v8.0 calibration; this is an
intentional uncalibrated leaderboard transfer probe for the v8.1 judge.

## Gate formula (unchanged architecture)

```
probe_z = (z_46 + z_40) / 2.0    # dual probe fused, per-family standardised
judge_z = (log_odds - mean) / sd   # standardised
score   = sigmoid(judge_z + cap × probe_z)
cap     = BASE_CAP + (judge_z × probe_z > 0) × (MAX_CAP − BASE_CAP)
```

## Weights required

```
submission/whitebox_probe/{qwen,gemma,nemotron}_probe/    # L46 weights (existing)
submission/whitebox_probe_L40/{qwen,gemma,nemotron}_probe/ # L40 weights (from legacy)
```

## Constants (frozen offline)

JUDGE_LOGIT_SD measured at 5.64 from 14 synthetic prompts on local GPU
(RTX 4090, Qwen3.5-9B + v8.0 adapter). It is intentionally reused without
recalibration for the v8.1 adapter and can still be overridden via env var.
BASE_CAP = 1 step = 0.125 / 5.638 = 0.022170
MAX_CAP  = 4 steps = 0.088681


In [ ]:
import os, sys, json
from pathlib import Path

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None
NNSIGHT_REMOTE = os.environ.get("NNSIGHT_REMOTE", "1").lower() in {"1", "true", "yes"}
THRESHOLD = float(os.environ.get("SUBMISSION_THRESHOLD", "0.5"))

import time
# v2.3.5: the sandbox enforces a single wall-clock budget per (notebook,
# dataset) run -- it SIGKILLs the process group at NOTEBOOK_BUDGET seconds
# and the first such failure aborts the WHOLE submission. NB_START anchors
# elapsed time so the judge retry can verify there is room for a second
# attempt before firing (see cell 11).
NB_START = time.time()
NOTEBOOK_BUDGET = float(os.environ.get("NOTEBOOK_BUDGET_SECONDS", "1800"))

print("method = sonic_v8.0.1")
print(f"dataset = {DATASET_NAME}")
print(f"limit   = {LIMIT}")
print(f"remote  = {NNSIGHT_REMOTE}")
print(f"threshold = {THRESHOLD}")

# v3: the judge now runs on EVERY dataset. v2.3.x decided that from the
# dataset name prefix ("validation-"), which is fail-closed: if the final
# held-out datasets carry any other prefix, the judge would silently not run
# on exactly the datasets that count. The direct-logit judge is one forward
# pass per row, so the compute it saved is no longer worth that risk, and the
# method is now identical on every dataset.


In [ ]:
import numpy as np
import torch
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "submission")
import util

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

In [ ]:
# Wrapped: if dataset loading fails, set base_model=None so later cells are skipped
try:
    # Load the dataset and pick the matching probe weights by base model
    from datasets import load_dataset
    ds = load_dataset(DATASET_NAME, split="test")
    if LIMIT:
        ds = ds.select(range(LIMIT))
    print(f"Loaded {len(ds)} examples")
    
    model_id = ds[0]["model"]
    lora = ds[0].get("lora", None)
    print(f"model = {model_id}")
    print(f"lora  = {lora}")
    
    base_model = None
    for family in ("gemma", "qwen", "nemotron"):
        if family in model_id.lower():
            base_model = family
            break
    if base_model is None:
        print(f"WARNING: no probe weights for base model {model_id}; "
              f"the judge carries this dataset alone")
    else:
        probe_dir = Path(f"submission/whitebox_probe/{base_model}_probe")
        print(f"base_model = {base_model}")
        print(f"probe_dir  = {probe_dir}")

    # v2.3 change 6: row ids and defaults that do NOT depend on the probe path.
    # A base model with no probe weights must still reach the judge, which is
    # black-box and needs no activations. The probe cells overwrite these.
    indices = [example.get("index", i) for i, example in enumerate(ds)]
    probe_scores = None
    probe_logits = None
    config = {}
except Exception as _cell_err:
    print(f"[FATAL] dataset loading failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
    base_model = None
    model_id = "unknown"
    lora = None
    ds = None
    indices = []
    probe_scores = None
    probe_logits = None
    config = {}

In [ ]:
if base_model is not None:
    try:
        # Load L46 probe config, weights, and standardization moments
        probe_dir_46 = Path(f"submission/whitebox_probe/{base_model}_probe")
        with open(probe_dir_46 / "config.json") as f:
            config_46 = json.load(f)
        feature_mean_46 = torch.load(probe_dir_46 / "feature_mean.pt", map_location=device)
        feature_std_46 = torch.load(probe_dir_46 / "feature_std.pt", map_location=device)
        print(f"L46 hidden_dim = {config_46['hidden_dim']}")
        print(f"L46 layer      = {config_46['layer']}")

        # v4: also load L40 probe (trained with same recipe, shared trunk, balanced)
        probe_dir_40 = Path(f"submission/whitebox_probe_L40/{base_model}_probe")
        with open(probe_dir_40 / "config.json") as f:
            config_40 = json.load(f)
        feature_mean_40 = torch.load(probe_dir_40 / "feature_mean.pt", map_location=device)
        feature_std_40 = torch.load(probe_dir_40 / "feature_std.pt", map_location=device)
        print(f"L40 hidden_dim = {config_40['hidden_dim']}")
        print(f"L40 layer      = {config_40['layer']}")
    except Exception as _cell_err:
        print(f"[FATAL] probe config loading failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
        base_model = None

In [ ]:
if base_model is not None:
    try:
        # Transformer token probe definition (must match training)
        # v3: the nemotron and qwen weight files are cut from a shared trunk
        # trained across all three families (gemma is unchanged). The class
        # below is untouched -- the export produces the same state_dict keys,
        # so loading is identical. See docs/sonic/sonic_v3.md section 4.
        import math

        def sinusoidal_position_encoding(seq_len, d_model, device=None):
            position = torch.arange(seq_len, dtype=torch.float32, device=device).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32, device=device)
                                 * (-math.log(10000.0) / d_model))
            enc = torch.zeros(seq_len, d_model, device=device)
            enc[:, 0::2] = torch.sin(position * div_term)
            cc = enc[:, 1::2].shape[1]
            enc[:, 1::2] = torch.cos(position * div_term)[:, :cc]
            return enc

        class TransformerTokenProbe(torch.nn.Module):
            def __init__(self, hidden_dim, d_model=128, n_heads=4, dim_feedforward=256, n_blocks=2, dropout=0.1):
                super().__init__()
                self.d_model = d_model
                self.projection = torch.nn.Linear(hidden_dim, d_model)
                block = torch.nn.TransformerEncoderLayer(
                    d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
                    dropout=dropout, batch_first=True)
                self.encoder = torch.nn.TransformerEncoder(block, num_layers=n_blocks)
                self.head = torch.nn.Sequential(torch.nn.Dropout(dropout), torch.nn.Linear(d_model, 1))
            def forward(self, padded_tokens, padding_mask):
                seq_len = padded_tokens.shape[1]
                pe = sinusoidal_position_encoding(seq_len, self.d_model, device=padded_tokens.device)
                x = self.projection(padded_tokens) + pe.unsqueeze(0)
                x = self.encoder(x, src_key_padding_mask=~padding_mask)
                m = padding_mask.unsqueeze(-1).to(x.dtype)
                pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
                return self.head(pooled).squeeze(-1)

        # L46 probe
        probe_46 = TransformerTokenProbe(
            hidden_dim=config_46['hidden_dim'],
            d_model=config_46['d_model'],
            n_heads=config_46['n_heads'],
            dim_feedforward=config_46['dim_feedforward'],
            n_blocks=config_46['n_blocks'],
            dropout=config_46['dropout'],
        ).to(device)
        probe_46.load_state_dict(torch.load(probe_dir_46 / "model.pt", map_location=device))
        probe_46.eval()

        # L40 probe
        probe_40 = TransformerTokenProbe(
            hidden_dim=config_40['hidden_dim'],
            d_model=config_40['d_model'],
            n_heads=config_40['n_heads'],
            dim_feedforward=config_40['dim_feedforward'],
            n_blocks=config_40['n_blocks'],
            dropout=config_40['dropout'],
        ).to(device)
        probe_40.load_state_dict(torch.load(probe_dir_40 / "model.pt", map_location=device))
        probe_40.eval()
        print("Both probes loaded and in eval mode.")
    except Exception as _cell_err:
        print(f"[FATAL] probe building failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
        base_model = None

In [ ]:
if base_model is not None:
    try:
        # Build the nnsight model handle: config/tokenizer load locally, the
        # weights stay on NDIF when tracing remotely
        model = util.build_model(model_id, lora)
        tokenizer = model.tokenizer
        print(f"Model loaded: {type(model).__name__}")
    except Exception as _cell_err:
        print(f"[FATAL] model building failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
        base_model = None

In [ ]:
if base_model is not None:
    try:
        # Locate the probed decoder layers; batch sizing comes from the probe config
        # (large models with little deployment headroom need smaller traces)
        layer_modules = util.decoder_layers(model)
        layer_idx_46 = min(config_46['layer'], len(layer_modules) - 1)
        layer_idx_40 = min(config_40['layer'], len(layer_modules) - 1)
        print(f"Decoder layers: {len(layer_modules)}, "
              f"using L46={layer_idx_46} L40={layer_idx_40}")

        PAD_ID = (tokenizer.pad_token_id if tokenizer.pad_token_id is not None
                  else tokenizer.eos_token_id)
        BATCH_TOKEN_BUDGET = config_46.get("extract_token_budget", 2560)
        MAX_BATCH_ROWS = config_46.get("extract_max_batch", 32)
        print(f"extraction batches: token budget {BATCH_TOKEN_BUDGET}, "
              f"max {MAX_BATCH_ROWS} rows")
    except Exception as _cell_err:
        print(f"[FATAL] layer finding failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
        base_model = None

In [ ]:
if base_model is not None:
    try:
        # Tokenize everything, compute response spans, build batches
        token_lists, spans, indices = [], [], []
        for i, example in enumerate(ds):
            token_ids, span = util.chat_preprocess(example["messages"], tokenizer, max_len=0)
            token_lists.append(token_ids)
            spans.append(span)
            indices.append(example.get("index", i))

        # Length-sorted batch packing under the token budget and row cap
        lengths = [len(t) for t in token_lists]
        order = sorted(range(len(lengths)), key=lambda p: lengths[p])
        batches, current = [], []
        for pos in order:
            if current and ((len(current) + 1) * lengths[pos] > BATCH_TOKEN_BUDGET
                            or len(current) >= MAX_BATCH_ROWS):
                batches.append(current); current = []
            current.append(pos)
        if current: batches.append(current)
        print(f"{len(token_lists)} examples, {len(batches)} batches")
    except Exception as _cell_err:
        print(f"[FATAL] tokenization failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
        base_model = None

In [ ]:
if base_model is not None:
    # Extract the probed layers' activations for every response token, all
    # batches bundled into one NDIF session.  v4: extract both L40 and L46 in
    # one trace, doubling the extraction volume (~2× hidden states per token).
    import time
    from contextlib import nullcontext

    def extract_activations():
        session = model.session(remote=True) if NNSIGHT_REMOTE else nullcontext()
        with session:
            pieces_46 = []
            pieces_40 = []
            for batch_positions in batches:
                batch_tokens = [token_lists[p] for p in batch_positions]
                batch_spans = [spans[p] for p in batch_positions]
                width = max(len(t) for t in batch_tokens)
                rows = len(batch_tokens)
                input_ids = torch.full((rows, width), PAD_ID, dtype=torch.long)
                attn_mask = torch.zeros(rows, width, dtype=torch.long)
                resp_mask = torch.zeros(rows, width, dtype=torch.bool)
                for row, (tokens, (start, end)) in enumerate(zip(batch_tokens, batch_spans)):
                    input_ids[row, :len(tokens)] = torch.tensor(tokens)
                    attn_mask[row, :len(tokens)] = 1
                    resp_mask[row, start:end] = True

                with model.trace({"input_ids": input_ids, "attention_mask": attn_mask}) as tracer:
                    h40 = layer_modules[layer_idx_40].output
                    h46 = layer_modules[layer_idx_46].output
                    if isinstance(h40, tuple):
                        h40 = h40[0]
                    if isinstance(h46, tuple):
                        h46 = h46[0]
                    mask_bool = resp_mask.to(h40.device)
                    sel40 = h40[mask_bool].to(torch.float16).detach().cpu().save()
                    sel46 = h46[mask_bool].to(torch.float16).detach().cpu().save()
                    tracer.stop()
                pieces_40.append(sel40)
                pieces_46.append(sel46)

            flat_40 = torch.cat(pieces_40, dim=0)
            flat_46 = torch.cat(pieces_46, dim=0)
            if NNSIGHT_REMOTE:
                flat_40 = flat_40.save()
                flat_46 = flat_46.save()
        # fp16 -> fp32 must happen in NUMPY on the client: the leaderboard
        # sandbox denies /proc/cpuinfo (Landlock) and torch's CPU half-precision
        # cast kernel hard-fails there ("Failed to initialize cpuinfo");
        # .numpy() is a zero-copy view and astype/clip run cpuinfo-free. The
        # clip also guards non-finite fp16 values from the download.
        finfo = np.finfo(np.float16)
        raw_40 = flat_40.cpu().numpy().astype(np.float32)
        raw_46 = flat_46.cpu().numpy().astype(np.float32)
        return (torch.from_numpy(np.clip(raw_40, finfo.min, finfo.max)),
                torch.from_numpy(np.clip(raw_46, finfo.min, finfo.max)))

    def is_transient(err):
        # EOFError is the corrupt-NDIF-download failure the organizers flagged;
        # the string markers catch dropped/streamed session transport errors.
        if isinstance(err, EOFError):
            return True
        markers = ("ran out of input", "eof", "connection", "reset", "timed out",
                   "timeout", "corrupt", "temporarily", "502", "503", "504")
        return any(m in str(err).lower() for m in markers)

    # v6-mini-long: record extraction time for cost tracking.
    extract_t0 = time.time()
    extract_seconds = None
    extraction_ok = False
    flat_features_40 = None
    flat_features_46 = None
    offsets = None
    MAX_ATTEMPTS = int(os.environ.get("EXTRACT_MAX_ATTEMPTS", "4"))
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            flat_40_batch, flat_46_batch = extract_activations()
            extraction_ok = True
            break
        except Exception as err:
            if attempt >= MAX_ATTEMPTS or not is_transient(err):
                import traceback as _tb
                _tb.print_exc()
                print(f"[FATAL] extraction failed after {attempt} attempt(s): {type(err).__name__}: {err}", flush=True)
                break
            wait = min(30, 2 ** attempt)
            print(f"transient extraction error on attempt {attempt}/{MAX_ATTEMPTS}: "
                  f"{type(err).__name__}: {err}; retrying in {wait}s")
            time.sleep(wait)
    extract_seconds = time.time() - extract_t0
    print(f"extraction: {extract_seconds:.0f}s, "
          f"{time.time() - NB_START:.0f}s since notebook start", flush=True)
    if extraction_ok:
        # Tokens arrive in batch-traversal order (batches are length-sorted); reorder
        # back to dataset order for scoring.
        span_lengths = [end - start for start, end in spans]
        batch_order = [p for batch in batches for p in batch]
        piece_lengths = [span_lengths[p] for p in batch_order]
        piece_offsets = np.cumsum([0] + piece_lengths).astype(np.int64)
        slot_of = {p: slot for slot, p in enumerate(batch_order)}

        def reorder(flat_batch):
            return torch.cat([
                flat_batch[piece_offsets[slot_of[p]]:piece_offsets[slot_of[p]] + span_lengths[p]]
                for p in range(len(spans))]).to(device)

        flat_features_46 = reorder(flat_46_batch)
        flat_features_40 = reorder(flat_40_batch)
        offsets = np.cumsum([0] + span_lengths).astype(np.int64)
        print(f"Extracted L46={flat_features_46.shape[0]} tokens, "
              f"L40={flat_features_40.shape[0]} tokens, "
              f"shape={tuple(flat_features_46.shape)}")
    else:
        print(f"[FALLBACK] using zero features (extraction failed)", file=sys.stderr)

In [ ]:
if base_model is not None and extraction_ok:
    try:
        # Score all examples with both probes
        def score_examples(flat_features, offsets, probe, feature_mean, feature_std, token_budget=8192):
            N = len(offsets) - 1
            lengths = (offsets[1:] - offsets[:-1]).tolist()
            order = sorted(range(N), key=lambda p: lengths[p])
            batches, current = [], []
            for pos in order:
                w = lengths[pos]
                if current and (len(current) + 1) * max(lengths[p] for p in current + [pos]) > token_budget:
                    batches.append(current); current = []
                current.append(pos)
            if current: batches.append(current)

            scores = np.zeros(N, dtype=np.float64)
            raw_logits = np.zeros(N, dtype=np.float64)
            with torch.no_grad():
                for row_ids in batches:
                    ml = max(lengths[r] for r in row_ids)
                    h = flat_features.shape[1]
                    padded = torch.zeros(len(row_ids), ml, h, dtype=torch.float32, device=device)
                    mask = torch.zeros(len(row_ids), ml, dtype=torch.bool, device=device)
                    for pos, row in enumerate(row_ids):
                        s, e = int(offsets[row]), int(offsets[row+1])
                        padded[pos, :e-s] = flat_features[s:e]
                        mask[pos, :e-s] = True
                    x = (padded - feature_mean) / feature_std
                    x = x * mask.unsqueeze(-1)
                    logits = probe(x, mask)
                    for pos, row in enumerate(row_ids):
                        # v3: keep the PRE-sigmoid score too. A float32
                        # sigmoid saturates to exactly 1.0 above a logit
                        # of ~17, tying every confident row together;
                        # AUROC ranks, so the blend uses the log-odds.
                        raw_logits[row] = float(logits[pos].item())
                        scores[row] = torch.sigmoid(logits[pos]).item()
            return scores, raw_logits

        probe_scores_46, probe_logits_46 = score_examples(flat_features_46, offsets, probe_46, feature_mean_46, feature_std_46)
        probe_scores_40, probe_logits_40 = score_examples(flat_features_40, offsets, probe_40, feature_mean_40, feature_std_40)
        print(f"L46 scored {len(probe_scores_46)} examples, range [{probe_scores_46.min():.4f}, {probe_scores_46.max():.4f}]")
        print(f"L40 scored {len(probe_scores_40)} examples, range [{probe_scores_40.min():.4f}, {probe_scores_40.max():.4f}]")

        # Standardise each probe's logits with frozen offline constants,
        # then average into one fused z-score (mean ~0, sd ~1).
        # v4: the scoring cell treats probe_z as pre-standardised
        # (probe_mean = 0.0, probe_sd = 1.0).
        PROBE_LOGIT_MEAN_46 = {'qwen': -0.367495, 'gemma': -2.523992, 'nemotron': -6.329599}
        PROBE_LOGIT_SD_46   = {'qwen': 6.626051,  'gemma': 4.510691,  'nemotron': 3.268104}
        PROBE_LOGIT_MEAN_40 = {'qwen': -0.1446,   'gemma': -1.1305,   'nemotron': -6.4427}
        PROBE_LOGIT_SD_40   = {'qwen': 6.9730,    'gemma': 6.3143,    'nemotron': 3.5210}
        DEFAULT_MEAN = -3.073695
        DEFAULT_SD = 4.801615

        mean_46 = PROBE_LOGIT_MEAN_46.get(base_model, DEFAULT_MEAN)
        sd_46 = PROBE_LOGIT_SD_46.get(base_model, DEFAULT_SD)
        mean_40 = PROBE_LOGIT_MEAN_40.get(base_model, DEFAULT_MEAN)
        sd_40 = PROBE_LOGIT_SD_40.get(base_model, DEFAULT_SD)

        z_46 = ((np.asarray(probe_logits_46, dtype=np.float64) - mean_46) / max(sd_46, 1e-8))
        z_40 = ((np.asarray(probe_logits_40, dtype=np.float64) - mean_40) / max(sd_40, 1e-8))
        probe_z_fused = (z_46 + z_40) / 2.0

        # Store as probe_logits so the scoring cell picks it up; also store
        # probe_scores for the probe-only fallback path.
        probe_logits = probe_z_fused
        probe_scores = torch.sigmoid(torch.from_numpy(probe_z_fused).float()).numpy()
        print(f"Fused probe_z: mean={probe_z_fused.mean():.4f} sd={probe_z_fused.std():.4f}, "
              f"range=[{probe_z_fused.min():.4f}, {probe_z_fused.max():.4f}]")
    except Exception as _cell_err:
        print(f"[FATAL] scoring failed: {type(_cell_err).__name__}: {_cell_err}", file=sys.stderr, flush=True)
        extraction_ok = False

if base_model is not None and not extraction_ok:
    # No probe features available — emit 0.5 scores
    print(f"[FALLBACK] scoring {len(indices)} examples with 0.5 (no extraction)", file=sys.stderr)
    probe_scores = np.full(len(indices), 0.5, dtype=np.float64)
    probe_logits = None
    print(f"Scored {len(probe_scores)} examples, range [{probe_scores.min():.4f}, {probe_scores.max():.4f}]")


In [ ]:
# Phoenix Wright v8.1 direct-margin judge — Kimi K3 Liars-enriched, single adapter.
# HP-KR + action routes, no intent adapter.  Softmax over binary label logits.
# Returns P(deception) in [0, 1].  We convert to log-odds for the sign gate.

judge_scores = None  # top-level: cell 12 falls back to probe-only if this stays None

try:
    from nnsight import LanguageModel
except ImportError:
    from nnsight.modeling.language import LanguageModel

import contextlib
from phoenix_wright import (
    DECEPTION_PROMPT,
    build_action_direct_prompt,
    build_epistemic_direct_prompt,
    build_primary_route_flags,
)

JUDGE_MODEL = "Qwen/Qwen3.5-9B"
MAIN_ADAPTER_REPO = os.environ.get(
    "PHOENIX_MAIN_ADAPTER_REPO",
    "Jazhyc/aletheias-phoenix-v8-1-kimi-k3-liars-full-r16-ep2",
)
MAX_PROMPT_TOKENS = int(os.environ.get("PHOENIX_MAX_PROMPT_TOKENS", "2048"))
BATCH_SIZE = int(os.environ.get("PHOENIX_BATCH_SIZE", "48"))
MEDIUM_BATCH_SIZE = int(os.environ.get("PHOENIX_MEDIUM_BATCH_SIZE", "32"))
MEDIUM_PROMPT_THRESHOLD = int(os.environ.get("PHOENIX_MEDIUM_PROMPT_THRESHOLD", "600"))
LONG_BATCH_SIZE = int(os.environ.get("PHOENIX_LONG_BATCH_SIZE", "16"))
LONG_PROMPT_THRESHOLD = int(os.environ.get("PHOENIX_LONG_PROMPT_THRESHOLD", "900"))
REMOTE_BATCHES_PER_SESSION = int(os.environ.get("PHOENIX_REMOTE_BATCHES_PER_SESSION", "0"))
NNSIGHT_LOCAL = os.environ.get("PHOENIX_NNSIGHT_LOCAL", "0").lower() in {"1", "true", "yes"}
JUDGE_RETRY_MARGIN = float(os.environ.get("JUDGE_RETRY_MARGIN_SECONDS", "180"))

print(f"judge=v8.1 kimi_k3_liars adapter={MAIN_ADAPTER_REPO} "
      f"batch={BATCH_SIZE}/{MEDIUM_BATCH_SIZE}/{LONG_BATCH_SIZE} "
      f"remote_batches_per_session={REMOTE_BATCHES_PER_SESSION}", flush=True)

MAX_CONTEXT_CHARS = 3000
MESSAGE_MAX_CHARS = 1500
OUTPUT_MAX_CHARS = 3000
TRUNCATION_MARKER = "\n\n[...truncated...]\n\n"
DIRECT_PREDICTION_PREFIX = "Prediction:"


def safe_text(value):
    return str(value).encode("utf-8", errors="replace").decode(
        "utf-8", errors="replace"
    )


def iter_messages(messages):
    if messages is None:
        return []
    if isinstance(messages, dict):
        return [messages]
    if isinstance(messages, (str, bytes)):
        return [{"role": "", "content": messages}]
    if isinstance(messages, (list, tuple)):
        return list(messages)
    return [{"role": "", "content": messages}]


def raw_message_field(message, field, default=""):
    if isinstance(message, dict):
        value = message.get(field, default)
    elif field == "content":
        value = message
    else:
        value = default
    return default if value is None else str(value)


def truncate_tail(text, max_chars):
    if max_chars <= 0 or len(text) <= max_chars:
        return text
    keep = max_chars - len(TRUNCATION_MARKER)
    if keep <= 0:
        return text[-max_chars:]
    return TRUNCATION_MARKER + text[-keep:]


def truncate_head_tail(text, max_chars):
    if max_chars <= 0 or len(text) <= max_chars:
        return text
    available = max_chars - len(TRUNCATION_MARKER)
    if available <= 0:
        return text[-max_chars:]
    head_chars = max(1, available // 3)
    tail_chars = available - head_chars
    return text[:head_chars] + TRUNCATION_MARKER + text[-tail_chars:]


def limited_message_field(message, field, max_chars):
    text = truncate_head_tail(raw_message_field(message, field), max_chars)
    return text.encode("utf-8", errors="replace").decode("utf-8")


def format_example(messages):
    normalized = iter_messages(messages)
    output = next(
        (
            limited_message_field(message, "content", OUTPUT_MAX_CHARS)
            for message in reversed(normalized)
            if limited_message_field(message, "role", 80) == "assistant"
        ),
        "",
    )
    context = "\n\n".join(
        f"{limited_message_field(message, 'role', 80).upper()}: "
        f"{limited_message_field(message, 'content', MESSAGE_MAX_CHARS)}"
        for message in normalized
    )
    return truncate_tail(context, MAX_CONTEXT_CHARS), output


def build_member_prompt(messages):
    context, output = format_example(messages)
    return (
        f"{DECEPTION_PROMPT}\n\n<context>\n{context}\n</context>"
        f"\n\n<output>\n{output}\n</output>"
    )


def build_direct_prompt(messages, tokenizer):
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": build_member_prompt(messages)}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return chat + DIRECT_PREDICTION_PREFIX


def binary_token_ids(tokenizer):
    ids = []
    for label in ("0", "1"):
        encoded = tokenizer.encode(label, add_special_tokens=False)
        if len(encoded) != 1:
            raise ValueError(
                f"binary label {label!r} tokenized as {encoded}; expected one token"
            )
        ids.append(int(encoded[0]))
    if len(set(ids)) != 2:
        raise ValueError(f"binary labels must have distinct token ids, got {ids}")
    return ids[0], ids[1]


print(f"judge config: max_prompt_tokens={MAX_PROMPT_TOKENS} "
      f"retry_margin={JUDGE_RETRY_MARGIN}s", flush=True)

for _judge_attempt in range(2):
    _judge_t0 = time.time()
    try:
        # Check we have budget for a full judge run
        _elapsed = time.time() - NB_START
        if _judge_attempt > 0:
            _remaining = NOTEBOOK_BUDGET - _elapsed
            if _remaining < JUDGE_RETRY_MARGIN:
                print(f"judge: insufficient budget for retry "
                      f"({_remaining:.0f}s < {JUDGE_RETRY_MARGIN}s margin), "
                      f"keeping previous attempt", flush=True)
                break

        # Load messages from dataset
        _examples = util.load_examples(DATASET_NAME)
        _n = len(_examples) if LIMIT is None else min(LIMIT, len(_examples))
        _messages = list(_examples["messages"][:_n])
        _indices = list(_examples["index"][:_n])
        harm_routes, action_routes = build_primary_route_flags(_messages)
        print(
            f"judge: loaded={len(_examples)} scoring={_n} "
            f"hpkr_routes={int(harm_routes.sum())} "
            f"action_routes={int(action_routes.sum())}",
            flush=True,
        )

        # Load judge model
        _model_kwargs = (
            {"device_map": "auto", "dispatch": True, "dtype": "bfloat16"}
            if NNSIGHT_LOCAL
            else {}
        )
        active_model = LanguageModel(
            JUDGE_MODEL,
            peft=MAIN_ADAPTER_REPO,
            **_model_kwargs,
        )
        active_tokenizer = active_model.tokenizer
        active_tokenizer.padding_side = "left"
        active_tokenizer.truncation_side = "left"
        if active_tokenizer.pad_token_id is None:
            active_tokenizer.pad_token = active_tokenizer.eos_token
        label_ids = list(binary_token_ids(active_tokenizer))
        print(
            f"judge: model loaded, binary_token_ids={label_ids} "
            f"pad_token_id={active_tokenizer.pad_token_id}",
            flush=True,
        )

        # Build prompts (HP-KR → epistemic, action → action, ordinary → default)
        _prompts = []
        for _pos, _row_msgs in enumerate(_messages):
            if harm_routes[_pos]:
                _builder = build_epistemic_direct_prompt
            elif action_routes[_pos]:
                _builder = build_action_direct_prompt
            else:
                _builder = build_direct_prompt
            _prompts.append(_builder(_row_msgs, active_tokenizer))

        # Batch and score
        _prompt_lengths = [
            len(active_tokenizer.encode(p, add_special_tokens=False))
            for p in _prompts
        ]
        _order = np.argsort(_prompt_lengths)
        _position_batches = []
        _cursor = 0
        while _cursor < len(_order):
            _cap = BATCH_SIZE
            _candidate = _order[_cursor:min(_cursor + _cap, len(_order))]
            _longest = max(_prompt_lengths[_p] for _p in _candidate)
            if _longest > MEDIUM_PROMPT_THRESHOLD:
                _cap = min(_cap, MEDIUM_BATCH_SIZE)
                _candidate = _order[_cursor:min(_cursor + _cap, len(_order))]
                _longest = max(_prompt_lengths[_p] for _p in _candidate)
            if _longest > LONG_PROMPT_THRESHOLD:
                _cap = min(_cap, LONG_BATCH_SIZE)
                _candidate = _order[_cursor:min(_cursor + _cap, len(_order))]
            _position_batches.append(_candidate.tolist())
            _cursor += len(_candidate)

        _encoded_batches = []
        for _positions in _position_batches:
            _encoded = active_tokenizer(
                [_prompts[_p] for _p in _positions],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_PROMPT_TOKENS,
            )
            _encoded_batches.append((_encoded, _positions, _encoded["input_ids"].shape[1]))

        _batches_per_session = (
            len(_encoded_batches)
            if NNSIGHT_LOCAL or REMOTE_BATCHES_PER_SESSION <= 0
            else REMOTE_BATCHES_PER_SESSION
        )
        _saved_groups = []
        for _group_start in range(0, len(_encoded_batches), _batches_per_session):
            _group_stop = min(_group_start + _batches_per_session, len(_encoded_batches))
            _score_pieces = []
            _session = contextlib.nullcontext() if NNSIGHT_LOCAL else active_model.session(remote=True)
            _shapes = [
                (len(_pos), _tokens)
                for _, _pos, _tokens in _encoded_batches[_group_start:_group_stop]
            ]
            print(
                f"judge: batches {_group_start + 1}-{_group_stop}/"
                f"{len(_encoded_batches)} shapes={_shapes}",
                flush=True,
            )
            with _session:
                for _encoded, _, _ in _encoded_batches[_group_start:_group_stop]:
                    with active_model.trace({
                        "input_ids": _encoded["input_ids"],
                        "attention_mask": _encoded["attention_mask"],
                        "logits_to_keep": 1,
                    }):
                        _label_logits = active_model.output.logits[:, -1, label_ids].float()
                        _piece = torch.softmax(_label_logits, dim=-1)[:, 1].detach().cpu()
                        _score_pieces.append(_piece)
                _group_scores = torch.cat(_score_pieces, dim=0).save()
            _saved_groups.append(_group_scores)

        _sorted_scores = torch.cat(_saved_groups, dim=0).float().numpy()
        judge_scores = np.empty(len(_prompts), dtype=float)
        _cursor2 = 0
        for _, _positions, _ in _encoded_batches:
            _count = len(_positions)
            judge_scores[_positions] = _sorted_scores[_cursor2:_cursor2 + _count]
            _cursor2 += _count
        judge_scores = np.clip(judge_scores, 0.0, 1.0)

        _jd = time.time() - _judge_t0
        print(
            f"judge: {len(judge_scores)} rows scored in {_jd:.0f}s, "
            f"range [{judge_scores.min():.4f}, {judge_scores.max():.4f}], "
            f"unique={len(np.unique(judge_scores))}",
            flush=True,
        )
        break
    except Exception as _judge_err:
        import traceback as _tb
        _tb.print_exc()
        print(
            f"judge attempt {_judge_attempt + 1}/2 failed: "
            f"{type(_judge_err).__name__}: {_judge_err}",
            file=sys.stderr, flush=True,
        )
        judge_scores = None
        if _judge_attempt >= 1:
            break
        time.sleep(5)

if judge_scores is None:
    print("judge: all attempts failed, gate will fall back to probe-only",
          file=sys.stderr, flush=True)
else:
    print(f"judge: final range [{judge_scores.min():.4f}, {judge_scores.max():.4f}] "
          f"mean={judge_scores.mean():.4f}", flush=True)


In [ ]:
# v4 sign gate — blend judge and probe under a capped nudge.
# Sonic v8.0.1 intentionally retains v8.0's Phoenix-judge calibration. When
# the v8.1 judge and probe agree on
# direction (judge_z × probe_z > 0), the cap opens to MAX_CAP; when they
# disagree, the probe is throttled to BASE_CAP.
# Linear contribution, no tanh.  Constants frozen offline.
#
# judge_z = (log_odds − JUDGE_LOGIT_MEAN) / JUDGE_LOGIT_SD   [sd ≈ 1]
# probe_z = pre-standardised (mean ≈ 0, sd ≈ 1, frozen per-family)
#
# step_z  = 0.125 / JUDGE_LOGIT_SD   (one bf16 quantization step)
# BASE_CAP = 1 × step_z   (probe can break ties, not reorder)
# MAX_CAP  = 4 × step_z   (probe can correct judge on Iris)

JUDGE_LOGIT_MEAN = float(os.environ.get("JUDGE_LOGIT_MEAN", "0.0"))
JUDGE_LOGIT_SD = float(os.environ.get("JUDGE_LOGIT_SD", "5.638"))

if probe_logits is not None and judge_scores is not None:
    try:
        probe_z = np.asarray(probe_logits, dtype=np.float64)
        judge_prob = np.asarray(judge_scores, dtype=np.float64)

        if len(probe_z) != len(indices) or len(judge_prob) != len(indices):
            raise ValueError(
                f"length mismatch: probe={len(probe_z)} judge={len(judge_prob)} "
                f"rows={len(indices)}"
            )

        # Convert judge probability → standardised log-odds
        _eps = 1e-8
        _clipped = np.clip(judge_prob, _eps, 1.0 - _eps)
        judge_log_odds = np.log(_clipped) - np.log(1.0 - _clipped)
        judge_z = (judge_log_odds - JUDGE_LOGIT_MEAN) / max(JUDGE_LOGIT_SD, 1e-8)

        # Cap constants: bf16 step = 0.125 in raw log-odds space
        _step_z = 0.125 / max(JUDGE_LOGIT_SD, 1e-8)
        BASE_CAP = 1.0 * _step_z   # 1 step
        MAX_CAP  = 4.0 * _step_z   # 4 steps
        PROBE_GAIN = 1.0            # pre-standardised probe_z

        # Sign-agreement gate
        agreement = (judge_z * probe_z > 0).astype(np.float64)
        cap = BASE_CAP + agreement * (MAX_CAP - BASE_CAP)

        # Blend in log-odds space, convert to probability
        combined = judge_z + cap * PROBE_GAIN * probe_z
        final_scores = 1.0 / (1.0 + np.exp(-combined))

        print(
            f"gate: v8.0.1 sign cap (v8.0 calibration; uncalibrated for v8.1 judge), "
            f"{len(final_scores)} rows, "
            f"score range [{final_scores.min():.4f}, {final_scores.max():.4f}], "
            f"agreement={agreement.mean():.1%}, "
            f"JUDGE_LOGIT_SD={JUDGE_LOGIT_SD:.1f} "
            f"step_z={_step_z:.6f} BASE_CAP={BASE_CAP:.6f} MAX_CAP={MAX_CAP:.6f} "
            f"judge_z range [{judge_z.min():.3f}, {judge_z.max():.3f}] "
            f"probe_z range [{probe_z.min():.3f}, {probe_z.max():.3f}]",
            flush=True,
        )
    except Exception as _cell_err:
        print(
            f"[WARN] gate failed: {type(_cell_err).__name__}: {_cell_err}",
            file=sys.stderr, flush=True,
        )
        final_scores = None
else:
    final_scores = None

# Fallback: if the judge is unavailable, use the probe alone.
if final_scores is None:
    if probe_scores is not None and len(probe_scores) == len(indices):
        final_scores = np.asarray(probe_scores, dtype=np.float64)
        print("gate: fallback to probe-only scoring", flush=True)
    else:
        final_scores = np.full(len(indices) if indices else 0, 0.5, dtype=np.float64)
        print("gate: fallback to 0.5 (no probe, no judge)", flush=True)


In [ ]:
if ds is not None:
    try:
        final_scores_arr = np.asarray(final_scores, dtype=np.float64)
        final_threshold = float(os.environ.get('SUBMISSION_THRESHOLD', '0.5'))
        print(f'submission: {len(final_scores_arr)} rows, '
              f'score range [{final_scores_arr.min():.4f}, {final_scores_arr.max():.4f}], '
              f'threshold={final_threshold}', flush=True)
        util.write_submission(indices if indices else [], final_scores_arr,
                              threshold=final_threshold)
    except Exception as _cell_err:
        print(f'[FATAL] write failed: {type(_cell_err).__name__}: {_cell_err}',
              file=sys.stderr, flush=True)
        util.write_submission(
            indices if indices else [],
            np.full(len(indices), 0.5, dtype=np.float64) if indices else np.zeros(0),
            threshold=0.5,
        )
else:
    util.write_submission([], np.zeros(0), threshold=0.5)
print('Done.')